In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## ಹೆಜ್ಜೆ 1: ರಚಿತ ಸರಣಿಗಳನ್ನು 위한 ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ವಿವರಿಸಿ

ಈ ಮಾದರಿಗಳು ಏಜೆಂಟ್ಗಳು ಹಿಂತಿರುಗಿಸುವ **ೕಖಲೆ** ನ್ನು ನಿರ್ವಚಿಸುತ್ತದೆ. ಪೈಡ್ಯಾಂಟಿಕ್ ಜೊತೆ `response_format` ಬಳಸಿ ಖಚಿತವಾಗುತ್ತದೆ:
- ✅ ಪ್ರಕಾರ-ಸುರಕ್ಷಿತ ಡೇಟಾ ತೆಗೆಯುವುದು
- ✅ ಸ್ವಯಂಚಾಲಿತ ಮಾನ್ಯತೆ
- ✅ ಮುಕ್ತ ಪಠ್ಯ ಉತ್ತರಗಳಿಂದ ನಿಖರತೆ ತಪ್ಪುಗಳು ಇಲ್ಲ
- ✅ ಕ್ಷೇತ್ರಗಳ ಆಧಾರದ ಮೇಲೆ ಸುಲಭ ಶರತಬಂಧಿತ ಮಾರ್ಗದರ್ಶನ


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## Step 2: ಹೋಟೆಲ್ ಬುಕ್ಕಿಂಗ್ ಸಾಧನವನ್ನು ಸೃಷ್ಟಿಸಿ

ಈ ಸಾಧನವೇ **availability_agent** ಕೋರುವ ಮತ್ತು ಕೊಠಡಿಗಳು ಲಭ್ಯವಿದೆಯೇ ಎಂದು ಪರಿಶೀಲಿಸುವುದಕ್ಕೆ ಬಳಸಲಾಗುತ್ತದೆ. ನಾವು `@ai_function` ಅಲಂಕಾರವನ್ನು ಉಪಯೋಗಿಸುತ್ತೇವೆ:
- ಪೈಥಾನ್ ಫಂಕ್ಶನ್ ಅನ್ನು AI-ಕರೆ ಮಾಡಲು ಸಾಧ್ಯವಾಗುವ ಸಾಧನವಾಗಿ ಪರಿವರ್ತಿಸಲು
- LLM ಗಾಗಿ ಸ್ವಯಂಚಾಲಿತವಾಗಿ JSON ಸ್ಕೀಮಾ ತಯಾರಿಸಲು
- ಪ್ಯಾರಾಮೀಟರ್ ಮಾನ್ಯತೆ ನಿರ್ವಹಿಸಲು
- ಏಜೆಂಟ್ಗಳಿಂದ ಸ್ವಯಂಚಾಲಿತ ಕರೆ ಮಾಡಲು ಸಾಧ್ಯವಾಗಿಸಲು

ಈ ಡೆಮೋಗೆ:
- **ಸ್ಟاڪ್ಹೋಮ್, ಸಿಯಾಟಲ್, ಟೋಕಿಯೋ, ಲಂಡನ್, ಆಂಸ್ಟರ್ಡ್ಯಾಮ್** → ಕೊಠಡಿಗಳು ✅
- **ಇತರೆ ಎಲ್ಲಾ ನಗರಗಳು** → ಕೊಠಡಿಗಳಿಲ್ಲ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## ಹಂತ 3: ಮಾರ್ಗವನ್ನು ನಿಗದಿಪಡಿಸಲು ಷರತ್ತು ಕಾರ್ಯಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ

ಈ ಕಾರ್ಯಗಳು ಏಜೆಂಟ್‌ನ ಪ್ರತಿಕ್ರಿಯೆಯನ್ನು ಪರಿಶೀಲಿಸಿ, ವರ್ಕ್ಫ್ಲೋದಲ್ಲ ಯಾವ ಮಾರ್ಗವನ್ನು ತೆಗೆದುಕೊಳ್ಳಬೇಕೋ ನಿರ್ಧಾರ ಮಾಡುತ್ತವೆ.

**ಪ್ರಮುಖ ಮಾದರಿ:**
1. ಸಂದೇಶವು `AgentExecutorResponse` ಇದೆಯೇ ಎಂದು ಪರಿಶೀಲಿಸಿ
2. ರಚಿತ ಔಟ್ಪುಟ್‌ನನ್ನು (Pydantic ಮಾದರಿ) ವಿಸ್ತರಿಸಿ
3. ಮಾರ್ಗವನ್ನು ನಿಯಂತ್ರಿಸಲು `True` ಅಥವಾ `False` ಅನ್ನು ಮರಳಿಸಿ

ಈ ಷರತ್ತುಗಳನ್ನು ವರ್ಕ್ಫ್ಲೋವು **ಧಾರಗಳಲ್ಲಿ** ಮೌಲ್ಯಮಾಪನ ಮಾಡಿ ಯಾವುದಾದರೂ ಕಾರ್ಯಾಚರಣೆಯನ್ನು ಮುಂದುವರಿಸಲು ನಿರ್ಧರಿಸುತ್ತದೆ.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## ಹಂತ 4: ಕಸ್ಟಮ್ ಡಿಸ್ಪ್ಲೇ ಕಾರ್ಯನಿರ್ವಹಣೆಯನ್ನು ರಚಿಸಿ

ಕಾರ್ಯನಿರ್ವಹಣೆಗಾರರು ಪರಿವರ್ತನೆಗಳು ಅಥವಾ ಪಾರ್ಶ್ವ ಪರಿಣಾಮಗಳನ್ನು ನಿರ್ವಹಿಸುವ ಕಾರ್ಯಪ್ರವಾಹ ಘಟಕಗಳಾಗಿದ್ದಾರೆ. ನಾವು ಅಂತಿಮ ಫಲಿತಾಂಶವನ್ನು ಪ್ರದರ್ಶಿಸುವ ಕಸ್ಟಮ್ ಕಾರ್ಯನಿರ್ವಹಣೆಯನ್ನು ರಚಿಸಲು `@executor` ಅಲಂಕರಣವನ್ನು ಬಳಸುತ್ತೇವೆ.

**ಪ್ರಮುಖ ಪರಿಕಲ್ಪನೆಗಳು:**
- `@executor(id="...")` - ಕಾರ್ಯನಿರ್ವಹಣೆಗಾರರಾಗಿ ಕಾರ್ಯವನ್ನು ಅಭ್ಯರ್ಥಿತಗೊಳಿಸುತ್ತದೆ
- `WorkflowContext[Never, str]` - ಇನ್‌ಪುಟ್/ಔಟ್‌ಪುಟ್‌ಗೆ ಪಡೆಸುವ ತರಗತಿಮುಹೂರ್ತಿ ಸೂಚನೆಗಳು
- `ctx.yield_output(...)` - ಅಂತಿಮ ಕಾರ್ಯಪ್ರವಾಹ ಫಲಿತಾಂಶವನ್ನು ನೀಡುತ್ತದೆ


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## ಹಂತ 5: ಪರಿಸರವಿನಿಯ ವೇರಿಯಬಲ್ಗಳು ಲೋಡ್ ಮಾಡಿ

LLM ಕ್ಲೈಎಂಟ್ ಅನ್ನು ಸಂರಚಿಸಿ. ಈ ಉದಾಹರಣೆಯು ಕೆಳಗಿನlarla ಕೆಲಸ ಮಾಡುತ್ತದೆ:
- **GitHub ಮಾದರಿಗಳು** (GitHub ಟೋಕನಿನೊಂದಿಗೆ ಉಚಿತ ಪದವಿ)
- **ಅಜೂರ್ OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಹಂತ 6: ರಚನೆಯಾದ ಮಾಹಿತಿ ಜೊತೆಗೆ AI ಏಜೆಂಟ್ಗಳನ್ನು ಸೃಷ್ಟಿಸಿ

ನಾವು **ಮೂರು ವಿಶಿಷ್ಟ ಏಜೆಂಟ್ಗಳನ್ನು** ರಚಿಸುತ್ತೇವೆ, ಪ್ರತಿಯೊಂದು `AgentExecutor` ನಲ್ಲಿ ಒಳಗೊಳ್ಳುತ್ತದೆ:

1. **availability_agent** - ಟೂಲ್ನಿಂದ ಹೋಟೆಲ್ ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸುತ್ತದೆ
2. **alternative_agent** - ಬದಲಾವಣೆ ಆಗಬಹುದಾದ ನಗರಗಳನ್ನು ಸೂಚಿಸುತ್ತದೆ (ಕೊಠಡಿಗಳು ಇಲ್ಲದಾಗ)
3. **booking_agent** - ಕಾಯ್ದಿರಿಸುವಿಕೆಯನ್ನು ಪ್ರೇರೇಪಿಸುತ್ತದೆ (ಕೊಠಡಿಗಳು ಲಭ್ಯವಿದ್ದಾಗ)

**ಮುಖ್ಯ ಲಕ್ಷಣಗಳು:**
- `tools=[hotel_booking]` - ಏಜೆಂಟ್ಗೆ ಟೂಲನ್ನು ಒದಗಿಸುತ್ತದೆ
- `response_format=PydanticModel` - ರಚನೆಯಾದ JSON ನಿರ್ಗಮನವನ್ನು ಜೋರಾಗಿ ಕೇಳಿಸುತ್ತದೆ
- `AgentExecutor(..., id="...")` - ಕಾರ್ಯಪ್ರವಾಹ ಬಳಕೆಗೆ ಏಜೆಂಟ್ನನ್ನು ಆವೃತ್ತಿಮಾಡುತ್ತದೆ


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## ಹಂತ 7: ಸಶರತ್ತು ಎಜ್‌ಗಳನ್ನು ಹೊಂದಿರುವ ವರ್ಕ್‌ಫ್ಲೋ ನಿರ್ಮಿಸಿ

ಇದೀಗ ನಾವು ಸರಣಿ ಮಾರ್ಗದರ್ಶನವನ್ನು ಹೊಂದಿರುವ ಗ್ರಾಫ್ ಅನ್ನು ನಿರ್ಮಿಸಲು `WorkflowBuilder` ಅನ್ನು ಬಳಸುತ್ತೇವೆ:

**ವರ್ಕ್‌ಫ್ಲೋ ರಚನೆ:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ಪ್ರಮುಖ ವಿಧಾನಗಳು:**
- `.set_start_executor(...)` - ಪ್ರಾರಂಭ ಬಿಂದುವನ್ನು ಹೊಂದಿಸುತ್ತದೆ
- `.add_edge(from, to, condition=...)` - ಸಶರತ್ತು ಎಜ್ ಸೇರಿಸುತ್ತದೆ
- `.build()` - ವರ್ಕ್‌ಫ್ಲೋ ಕೊನೆಗೊಳಿಸುತ್ತದೆ


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## ಹಂತ 8: ಟೆಸ್ಟ್‌ ಕೇಸ್ 1 ಅನ್ನು ನಡೆಸಿ - ಲಭ್ಯತೆ ಇಲ್ಲದ ನಗರ (ಪ್ಯಾರಿಸ್)

ನಮ್ಮ ಅನುಕರಣೆಯಲ್ಲಿ ಕೋಣೆಗಳಿಲ್ಲದ ಪ್ಯಾರಿಸ್‌ನಲ್ಲಿ ಹೋಟೆಲ್‌ಗಳನ್ನು ವಿನಂತಿಸುವ ಮೂಲಕ **ಲಭ್ಯತೆ ಇಲ್ಲದ** ಮಾರ್ಗವನ್ನು ಪರೀಕ್ಷಿಸೋಣ.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: ಟೆಸ್ಟ್ ಕೇಸ್ 2 ರನ್ ಮಾಡಿ - ಲಭ್ಯತೆ ಜೊತೆಗೆ ನಗರ (ಸ್ಟಾಕ್‌ಹೋಮ್)

ಈಗ ನಾವು **ಲಭ್ಯತೆ** ಮಾರ್ಗವನ್ನು ಪರೀಕ್ಷಿಸೋಣ ಸ್ಟಾಕ್‌ಹೋಮ್‌ನ ಹೋಟೆಲ್‌ಗಳನ್ನು ಕೇಳಿ (ನಮ್ಮ ಸಿಮ್ಯುಲೇಶನಿನಲ್ಲಿ ಕೊಠಡಿಗಳು ಇರುವ शहर).


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ಪ್ರಮುಖ ಸೆರೆಹಿಡಿತಗಳು ಮತ್ತು ಮುಂದಿನ ಹೆಜ್ಜೆಗಳು

### ✅ ನೀವು ಕಲಿತಿರುವುದು:

1. **WorkflowBuilder ಮಾದರಿ**
   - ಪ್ರವೇಶ ಬಿಂದುವನ್ನು ನಿರ್ಧರಿಸಲು `.set_start_executor()` ಅನ್ನು ಬಳಸಿ
   - الشرطي ರೌಟಿಂಗ್ ಗಾಗಿ `.add_edge(from, to, condition=...)` ಅನ್ನು ಬಳಸಿ
   - ವರ್ಕ್‌ಫ್ಲೋ ಅನ್ನು ಪೂರ್ಣಗೊಳಿಸಲು `.build()` ಅನ್ನು ಕರೆಸಿ

2. **شرطي ರೌಟಿಂಗ್**
   - ಶರತ್ತು ಕಾರ್ಯಗಳು `AgentExecutorResponse` ಡೇಟಾವನ್ನು ಪರಿಶೀಲಿಸುತ್ತವೆ
   - ರೌಟಿಂಗ್ ನಿರ್ಧಾರಗಳನ್ನು ಮಾಡಲು ರಚನೆಯಾದ output ಗಳನ್ನು ಪಾರ್ಸ್ ಮಾಡುವುದು
   - ಒಂದು ಎಡ್ಜ್ ಸಕ್ರಿಯಗೊಳಿಸಲು `True` ಮರಳಿಸಿ, ಅದನ್ನು ತೆರವುಗೊಳಿಸಲು `False` ಮರಳಿಸಿ

3. **ಟೂಲ್ ಏಕೀಕರಣ**
   - Python ಫಂಕ್ಷನ್‌ಗಳನ್ನು AI ಟೂಲ್ಗಳಾಗಿ ಪರಿವರ್ತಿಸಲು `@ai_function` ಅನ್ನು ಬಳಸಿ
   - ಅಗತ್ಯವಿದ್ದಾಗ ಏಜೆಂಟ್ಗಳು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಟೂಲ್ಗಳನ್ನು ಕರೆಯುತ್ತವೆ
   - ಟೂಲ್ಗಳು ಏಜೆಂಟ್‌ಗಳು ಪಾರ್ಸ್ ಮಾಡಬಹುದಾದ JSON ಅನ್ನು ಮರಳಿಸುತ್ತವೆ

4. **ಸಂರಚಿತ output ಗಳು**
   - ಪ್ರಕಾರ-ಸುರಕ್ಷಿತ ಡೇಟಾ ಎಕ್ಸ್ಟ್ರಾಕ್ಷನ್‌ಗಾಗಿ Pydantic ಮಾದರಿಗಳನ್ನು ಬಳಸಿ
   - ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸುವಾಗ `response_format=MyModel` ಅನ್ನು ಹೊಂದಿಸಿ
   - ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು `Model.model_validate_json()` ಮೂಲಕ ಪಾರ್ಸ್ ಮಾಡಿರಿ

5. **ಕಸ್ಟಮ್ ಎಕ್ಸಿಕ್ಯೂಟರ್ಸ್**
   - ವರ್ಕ್‌ಫ್ಲೋ ಘಟಕಗಳನ್ನು ರಚಿಸಲು `@executor(id="...")` ಅನ್ನು ಬಳಸಿ
   - ಎಕ್ಸಿಕ್ಯೂಟರ್ಸ್ ಡೇಟಾವನ್ನು ಪರಿವರ್ತಿಸಿದಾಗ ಅಥವಾ ಪಕ್ಕದ ಪರಿಣಾಮಗಳನ್ನು ಮಾಡಲು ಬಳಸಬಹುದು
   - ವರ್ಕ್‌ಫ್ಲೋ ಫಲಿತಾಂಶಗಳನ್ನು ಉತ್ಪಾದಿಸಲು `ctx.yield_output()` ಅನ್ನು ಬಳಸಿ

### 🚀 ನೈಜ ಜಗತ್ತಿನ ಅನ್ವಯಿಕೆಗಳು:

- **ಪ್ರಯಾಣ ಬುಕ್ಕಿಂಗ್**: ಲಭ್ಯತೆ ಪರಿಶೀಲಿಸಿ, ಪರ್ಯಾಯಗಳನ್ನು ಸೂಚಿಸಿ, ಆಯ್ಕೆಗಳು ಹೋಲಿಸಿ
- **ಗ್ರಾಹಕ ಸೇವೆ**: ಸಮಸ್ಯೆಯ ಪ್ರಕಾರ, ಮನೋಭಾವ, ಪ್ರಾಮುಖ್ಯತೆಯ ಆಧಾರದಲ್ಲಿ ಮಾರ್ಗಗುಮಾಸು
- **ಇ-ಕಾಮರ್ಸ್**: ಇನ್‌ವೆಂಟರಿ ಪರಿಶೀಲಿಸಿ, ಪರ್ಯಾಯ ಸೂಚಿಸಿ, ಆರ್ಡರ್ ಪ್ರಕ್ರಿಯೆ ಮಾಡಿ
- **ವಿಷಯ ನಿಯಂತ್ರಣ**: ವಿಷ ಕಾರ್ಯಕ್ಷಮತೆ ಅಂಕಗಳನ್ನು ಮತ್ತು ಬಳಕೆದಾರ ಧ್ವಜಗಳ ಆಧಾರದ ಮೇಲೆ ಮಾರ್ಗಗುಮಾಸು
- **ಅನುಮೋದನಾ ವರ್ಕ್‌ಫ್ಲೋಗಳು**: ಮೊತ್ತ, ಬಳಕೆದಾರ ಪಾತ್ರ, ಅಪಾಯ ಮಟ್ಟದ ಆಧಾರದಲ್ಲಿ ಮಾರ್ಗಗುಮಾಸು
- **ಬಹು-ಹಂತ ಪ್ರಕ್ರಿಯೆ**: ಡೇಟಾ ಗುಣಮಟ್ಟ, ಸಂಪೂರ್ಣತೆ ಆಧಾರದ ಮೇಲೆ ಮಾರ್ಗಗುಮಾಸು

### 📚 ಮುಂದಿನ ಹಂತಗಳು:

- ಹೆಚ್ಚು ಸಂಕೀರ್ಣ ಶರತ್ತುಗಳನ್ನು (ಬಹು ಮಾನದಂಡಗಳು) ಸೇರಿಸುವುದು
- ವರ್ಕ್‌ಫ್ಲೋ ಸ್ಥಿತಿಯ ನಿರ್ವಹಣೆಯೊಂದಿಗೆ ಲೂಪ್‌ಗಳನ್ನು ಜಾರಿಗೆ ತರುವಿಕೆ
- ಪುನಃಬಳಕೆ ಮಾಡಬಹುದಾದ ಘಟಕಗಳಿಗಾಗಿ ಉಪ-ವರ್ಕ್‌ಫ್ಲೋಗಳನ್ನು ಸೇರಿಸುವುದು
- ನೈಜ API ಗಳೊಂದಿಗೆ ಏಕೀಕರಣ (ಹೋಟೆಲ್ ಬುಕ್ಕಿಂಗ್, ಇನ್‌ವೆಂಟರಿ ವ್ಯವಸ್ಥೆಗಳು)
- ದೋಷ ನಿರ್ವಹಣೆ ಮತ್ತು ಬ್ಯಾಕಪ್ ಮಾರ್ಗಗಳನ್ನು ಸೇರಿಸುವುದು
- ನಿರ್ಮಿತ ದೃಶ್ಯೀಕರಣ ಸಾಧನಗಳೊಂದಿಗೆ ವರ್ಕ್‌ಫ್ಲೋಗಳನ್ನು ದೃಶ್ಯೀಕರಿಸುವುದು


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
